In [11]:
import cv2
import os

def separate_and_resize(input_image_paths, output_folders, target_size=(256, 256), background_ratio=0.2):
    for input_image_path, output_folder in zip(input_image_paths, output_folders):
        # Read the input image
        image = cv2.imread(input_image_path)

        # Convert the image to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply thresholding to create a binary image
        _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)  # Adjust the threshold value here

        # Find contours in the binary image
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Create the output folder if it doesn't exist
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        # Iterate through the contours and save each separated and resized picture
        for i, contour in enumerate(contours):
            # Get the bounding box of the contour
            x, y, w, h = cv2.boundingRect(contour)
            
            # Calculate the amount of background to add
            background_width = int(w * background_ratio)
            background_height = int(h * background_ratio)
            
            # Expand the bounding box to encompass more background
            x -= background_width
            y -= background_height
            w += 2 * background_width
            h += 2 * background_height
            
            # Ensure the bounding box is within image bounds
            x = max(x, 0)
            y = max(y, 0)
            w = min(w, image.shape[1])
            h = min(h, image.shape[0])
            
            # Crop the region of interest (ROI) from the original image
            roi = image[y:y+h, x:x+w]
            
            # Resize the ROI to the target size
            resized_roi = cv2.resize(roi, target_size)

            # Save the resized image to the output folder
            output_path = os.path.join(output_folder, f"picture_{i + 1}.png")
            cv2.imwrite(output_path, resized_roi)

if __name__ == "__main__":
    # Replace 'input_image_paths' with a list of input image paths
    input_image_paths = ['1 (1).png']
    
    # Replace 'output_folders' with a list of corresponding output folder paths
    output_folders = ['output_folder_final']

    separate_and_resize(input_image_paths, output_folders)


In [5]:
import cv2
import os
import numpy as np

def separate_and_resize(input_image_paths, output_folders, target_size=(4000, 4000), background_ratio=0.2):
    for input_image_path, output_folder in zip(input_image_paths, output_folders):
        # Read the input image
        image = cv2.imread(input_image_path)

        # Convert the image to grayscale
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply thresholding to create a binary image
        _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)  # Adjust the threshold value here

        # Find contours in the binary image
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Create the output folder if it doesn't exist
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        # Iterate through the contours and save each separated and resized picture
        for i, contour in enumerate(contours):
            # Get the bounding box of the contour
            x, y, w, h = cv2.boundingRect(contour)
            
            # Calculate the amount of background to add
            background_width = int(w * background_ratio)
            background_height = int(h * background_ratio)
            
            # Expand the bounding box to encompass more background
            x -= background_width
            y -= background_height
            w += 2 * background_width
            h += 2 * background_height
            
            # Ensure the bounding box is within image bounds
            x = max(x, 0)
            y = max(y, 0)
            w = min(w, image.shape[1])
            h = min(h, image.shape[0])
            
            # Crop the region of interest (ROI) from the original image
            roi = image[y:y+h, x:x+w]
            
            # Resize the ROI while preserving aspect ratio
            resized_roi = resize_with_aspect_ratio(roi, target_size)
            
            # Save the resized image to the output folder
            output_path = os.path.join(output_folder, f"picture_{i + 1}.png")
            cv2.imwrite(output_path, resized_roi)

def resize_with_aspect_ratio(image, target_size):
    """Resize image while preserving aspect ratio."""
    h, w = image.shape[:2]
    target_w, target_h = target_size

    # Calculate aspect ratio
    aspect_ratio = w / h

    # Determine resizing dimensions based on aspect ratio
    if aspect_ratio > 1:  # Landscape orientation
        new_w = target_w
        new_h = int(new_w / aspect_ratio)
    else:  # Portrait orientation
        new_h = target_h
        new_w = int(new_h * aspect_ratio)

    # Resize image
    resized_image = cv2.resize(image, (new_w, new_h))

    # Create a blank canvas of the target size
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    # Calculate position to paste the resized image on the canvas to center it
    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2

    # Paste the resized image onto the canvas
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_image

    return canvas

if __name__ == "__main__":
    # Replace 'input_image_paths' with a list of input image paths
    input_image_paths = ['1 (2).jpg', ]
    # Replace 'output_folders' with a list of corresponding output folder paths
    output_folders = ['2 (2)',]

    separate_and_resize(input_image_paths, output_folders)


KeyboardInterrupt: 

In [1]:
import cv2
import os
import numpy as np

def separate_and_resize(input_image_paths, output_folders, target_size=(4000, 4000), background_ratio=0.2, max_leaves=5):
    for input_image_path, output_folder in zip(input_image_paths, output_folders):
        image = cv2.imread(input_image_path)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)

        # Step 1: Threshold on brightness to get foreground (leaf)
        _, mask = cv2.threshold(v, 25, 255, cv2.THRESH_BINARY)

        # Step 2: Clean up mask
        kernel = np.ones((9, 9), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # Fill holes inside leaf
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)   # Remove small outside noise

        # Save cleaned mask for visual inspection
        os.makedirs(output_folder, exist_ok=True)
        cv2.imwrite(os.path.join(output_folder, "debug_mask_cleaned.png"), mask)

        # Step 3: Find contours and sort by size
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:max_leaves]

        for i, contour in enumerate(contours):
            area = cv2.contourArea(contour)
            if area < 1000:
                continue  # skip small noise

            # Bounding box + background padding
            x, y, w, h = cv2.boundingRect(contour)
            bg_w = int(w * background_ratio)
            bg_h = int(h * background_ratio)

            x = max(x - bg_w, 0)
            y = max(y - bg_h, 0)
            w = min(w + 2 * bg_w, image.shape[1] - x)
            h = min(h + 2 * bg_h, image.shape[0] - y)

            # Crop the region of interest
            roi = image[y:y+h, x:x+w]

            # --- Force background to black using mask ---
            roi_mask = np.zeros((h, w), dtype=np.uint8)
            contour_shifted = contour - [x, y]  # shift contour to cropped region
            cv2.drawContours(roi_mask, [contour_shifted], -1, 255, thickness=cv2.FILLED)

            roi_clean = cv2.bitwise_and(roi, roi, mask=roi_mask)

            # Resize while preserving aspect ratio
            resized_roi = resize_with_aspect_ratio(roi_clean, target_size)

            # Save final output
            output_path = os.path.join(output_folder, f"leaf_{i + 1}.png")
            cv2.imwrite(output_path, resized_roi)

def resize_with_aspect_ratio(image, target_size):
    h, w = image.shape[:2]
    target_w, target_h = target_size
    aspect_ratio = w / h

    if aspect_ratio > 1:
        new_w = target_w
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = target_h
        new_w = int(new_h * aspect_ratio)

    resized_image = cv2.resize(image, (new_w, new_h))
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_image

    return canvas


if __name__ == "__main__":
    input_image_paths = ['1 (41).jpg']  # Replace with your filename
    output_folders = ['41']
    separate_and_resize(input_image_paths, output_folders)


In [2]:
import cv2
import os
import numpy as np

def separate_and_resize(input_image_paths, output_folders, target_size=(4000, 4000), background_ratio=0.2, max_leaves=5):
    for input_image_path, output_folder in zip(input_image_paths, output_folders):
        image = cv2.imread(input_image_path)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        h, s, v = cv2.split(hsv)

        # Step 1: Threshold on brightness to get foreground (leaf)
        _, mask = cv2.threshold(v, 25, 255, cv2.THRESH_BINARY)

        # Step 2: Clean up mask
        kernel = np.ones((9, 9), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # Fill holes inside leaf
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)   # Remove small outside noise

        # Save cleaned mask for visual inspection
        os.makedirs(output_folder, exist_ok=True)
        cv2.imwrite(os.path.join(output_folder, "debug_mask_cleaned.png"), mask)

        # Step 3: Find contours and sort by size
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:max_leaves]

        for i, contour in enumerate(contours):
            area = cv2.contourArea(contour)
            if area < 1000:
                continue  # skip small noise

            # Bounding box + background padding
            x, y, w, h = cv2.boundingRect(contour)
            bg_w = int(w * background_ratio)
            bg_h = int(h * background_ratio)

            x = max(x - bg_w, 0)
            y = max(y - bg_h, 0)
            w = min(w + 2 * bg_w, image.shape[1] - x)
            h = min(h + 2 * bg_h, image.shape[0] - y)

            # Crop the region of interest
            roi = image[y:y+h, x:x+w]

            # --- Force background to black using mask ---
            roi_mask = np.zeros((h, w), dtype=np.uint8)
            contour_shifted = contour - [x, y]  # shift contour to cropped region
            cv2.drawContours(roi_mask, [contour_shifted], -1, 255, thickness=cv2.FILLED)

            roi_clean = cv2.bitwise_and(roi, roi, mask=roi_mask)

            # Resize while preserving aspect ratio
            resized_roi = resize_with_aspect_ratio(roi_clean, target_size)

            # Save final output
            output_path = os.path.join(output_folder, f"leaf_{i + 1}.png")
            cv2.imwrite(output_path, resized_roi)

def resize_with_aspect_ratio(image, target_size):
    h, w = image.shape[:2]
    target_w, target_h = target_size
    aspect_ratio = w / h

    if aspect_ratio > 1:
        new_w = target_w
        new_h = int(new_w / aspect_ratio)
    else:
        new_h = target_h
        new_w = int(new_h * aspect_ratio)

    resized_image = cv2.resize(image, (new_w, new_h))
    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_image

    return canvas

if __name__ == "__main__":
    input_image_paths = [f'1 ({i}).JPG' for i in range(1, 43)]
    output_folders = [str(i) for i in range(1, 43)]
    separate_and_resize(input_image_paths, output_folders)
